**QUESTIONS POUR LA CORRECTION**

exo1 : j'ai mis à -3 pour les 3mm derrière, juste ?

exo2 : quelle formule ils ont utilisé ? 

## Exercise 1: Odor from behind

In this exercise, let's simulate a scenario in which an attractive odor source is placed directly behind the fly. How do you expect the fly to behave in this case?

In [1]:
import numpy as np
from tqdm import trange

from flygym.compose import ActuatorType
from flygym.compose.world import FlatGroundWorld, OdorMixin
from flygym.examples.locomotion import TurningController
from utils import create_fly, create_simulation, show_video


class FlatGroundWorldWithOdorSources(OdorMixin, FlatGroundWorld):
    pass


def odor_intensity_to_control_signal(
    odor_intensities,
    attractive_gain=-500,
    aversive_gain=80,
):
    """Convert odor sensor readings to a turning control signal.

    Parameters
    ----------
    odor_intensities : np.ndarray
        Odor intensities from the four sensors, shape ``(4, n_odor_dims)``.
    attractive_gain : float
        Gain applied to the attractive odor dimension.
    aversive_gain : float
        Gain applied to the aversive odor dimension.

    Returns
    -------
    np.ndarray
        Control signal of shape ``(2,)`` for left and right descending drive.
    """
    attractive_intensities = np.average(
        odor_intensities[:, 0].reshape(2, 2), axis=0, weights=[9, 1]
    )
    aversive_intensities = np.average(
        odor_intensities[:, 1].reshape(2, 2), axis=0, weights=[10, 0]
    )
    attractive_bias = (
        attractive_gain
        * (attractive_intensities[0] - attractive_intensities[1])
        / attractive_intensities.mean()
        if attractive_intensities.mean() != 0
        else 0
    )
    aversive_bias = (
        aversive_gain
        * (aversive_intensities[0] - aversive_intensities[1])
        / aversive_intensities.mean()
        if aversive_intensities.mean() != 0
        else 0
    )
    effective_bias = aversive_bias + attractive_bias
    effective_bias_norm = np.tanh(effective_bias**2) * np.sign(effective_bias)
    assert np.sign(effective_bias_norm) == np.sign(effective_bias)

    control_signal = np.ones(2)
    side_to_modulate = int(effective_bias_norm > 0)
    modulation_amount = np.abs(effective_bias_norm) * 0.8
    control_signal[side_to_modulate] -= modulation_amount
    return control_signal


def run_olfaction_simulation(sim, fly, world, controller, run_time=20, decision_interval=0.05,
                             get_control_signal=odor_intensity_to_control_signal):
    """Run the olfaction simulation loop.

    Parameters
    ----------
    sim : Simulation
        The simulation instance.
    fly : Fly
        The fly instance.
    world : BaseWorld
        The world (must have ``odor_positions``).
    controller : TurningController
        The locomotion controller.
    run_time : float
        Total simulation time in seconds.
    decision_interval : float
        Time between control signal updates (seconds).
    get_control_signal : callable
        Function mapping odor intensities to a control signal.
    """
    num_decision_steps = int(run_time / decision_interval)
    physics_steps_per_decision = int(decision_interval / sim.timestep)

    with trange(num_decision_steps * physics_steps_per_decision) as pbar:
        for _ in range(num_decision_steps):
            odor_intensities = sim.get_olfaction(fly.name)
            control_signal = get_control_signal(odor_intensities)

            for _ in range(physics_steps_per_decision):
                joint_angles, adhesion = controller.step(control_signal)
                sim.set_actuator_inputs(fly.name, ActuatorType.POSITION, joint_angles)
                sim.set_actuator_inputs(fly.name, ActuatorType.ADHESION, adhesion)
                sim.step()
                sim.render_as_needed()

                fly_pos = sim.mj_data.body(f"{fly.name}/").xpos[:2].copy()
                if np.linalg.norm(fly_pos - world.odor_positions[0, :2]) < 2:
                    break

                pbar.update()

In [2]:
world = FlatGroundWorldWithOdorSources()

################################################################
# TODO: Create an odor arena with a single attractive odor source
# placed 3 mm behind the fly.
world.add_odor_source(
    pos=[-3,0,1.5],
    peak_intensity=[1.0, 0.0],
    marker_rgba=np.array([255, 127, 14, 255]) / 255,
)
################################################################

fly = create_fly(enable_olfaction=True)
sim = create_simulation(fly, world, camera_kwargs={"pos": (0, 0, 35)})
controller = TurningController(sim.timestep)

run_olfaction_simulation(sim, fly, world, controller, run_time=20)
show_video(sim)

 18%|█▊        | 35365/200000 [01:54<08:52, 309.12it/s]


birdeyecam This browser does not support the video tag.


## Exercise 2: Noisy sensors

Let's see what happens when the measurements of odor intensity are noisy. We subclass `Simulation` to inject multiplicative noise into the olfaction readout:

In [3]:
from functools import cached_property
from flygym import Simulation


class NoisySensorSimulation(Simulation):
    """Simulation subclass that injects multiplicative noise into olfaction."""

    @cached_property
    def rng(self):
        return np.random.default_rng(0)

    def get_olfaction(self, fly_name: str) -> np.ndarray:
        odor_intensities = super().get_olfaction(fly_name)
        noise_var = 5
        odor_intensities *= self.rng.gamma(
            shape=noise_var, scale=1 / noise_var, size=odor_intensities.shape
        )
        return odor_intensities

In [3]:
world = FlatGroundWorldWithOdorSources()
world.add_odor_source(
    pos=(3, 4, 1.5),
    peak_intensity=[1.0, 0.0],
    marker_rgba=np.array([255, 127, 14, 255]) / 255,
)
fly = create_fly(enable_olfaction=True)
sim = create_simulation(
    fly, world,
    camera_kwargs={"pos": (0, 0, 35)},
    simulation_class=NoisySensorSimulation,
)
controller = TurningController(sim.timestep)

run_olfaction_simulation(sim, fly, world, controller, run_time=2)
show_video(sim)

100%|██████████| 20000/20000 [01:02<00:00, 321.58it/s]


birdeyecam This browser does not support the video tag.


Let's implement a new control algorithm that smooths noisy odor readings before computing the steering signal. This should help the fly navigate toward the odor source despite noisy sensors.

In [4]:
world = FlatGroundWorldWithOdorSources()
world.add_odor_source(
    pos=(3, 4, 1.5),
    peak_intensity=[1.0, 0.0],
    marker_rgba=np.array([255, 127, 14, 255]) / 255,
)
fly = create_fly(enable_olfaction=True)
sim = create_simulation(
    fly, world,
    camera_kwargs={"pos": (0, 0, 35)},
    simulation_class=NoisySensorSimulation,
)

decision_interval = 0.05
run_time = 2
num_decision_steps = int(run_time / decision_interval)
physics_steps_per_decision = int(decision_interval / sim.timestep)

controller = TurningController(sim.timestep)
odor_intensities_smooth = None

with trange(num_decision_steps * physics_steps_per_decision) as pbar:
    for _ in range(num_decision_steps):
        odor_intensities = sim.get_olfaction(fly.name)
        if odor_intensities_smooth is None:
            odor_intensities_smooth = odor_intensities
        else:
            ############################################################
            # TODO: Calculate the exponential moving average of the
            # odor intensities.
            # Refer to https://en.wikipedia.org/wiki/Exponential_smoothing
            alpha = 0.001
            odor_intensities_smooth = alpha*odor_intensities+(1-alpha)*odor_intensities_smooth
            ############################################################
        control_signal = odor_intensity_to_control_signal(odor_intensities_smooth)

        for _ in range(physics_steps_per_decision):
            joint_angles, adhesion = controller.step(control_signal)
            sim.set_actuator_inputs(fly.name, ActuatorType.POSITION, joint_angles)
            sim.set_actuator_inputs(fly.name, ActuatorType.ADHESION, adhesion)
            sim.step()
            sim.render_as_needed()

            fly_pos = sim.mj_data.body(f"{fly.name}/").xpos[:2].copy()
            if np.linalg.norm(fly_pos - world.odor_positions[0, :2]) < 2:
                break

            pbar.update()

show_video(sim)

 43%|████▎     | 8542/20000 [00:25<00:33, 338.89it/s]


birdeyecam This browser does not support the video tag.
